In [1]:
import pandas as pd
import numpy as np
import os
import urllib.request
import zipfile
from huggingface_hub import hf_hub_download
from rich import print as rprint
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

os.makedirs("/kaggle/working/data", exist_ok = True)
os.makedirs("/kaggle/working/processed", exist_ok = True)

In [2]:
catalog_path = hf_hub_download(
    repo_id = "Subhadip007/UERP_Dataset",
    filename = "catalog_with_features_v1.parquet",
    repo_type = "dataset",
    token = HF_TOKEN,
)

catalog = pd.read_parquet(catalog_path)
rprint("Our catalog:", catalog.shape)

catalog_with_features_v1.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

Our catalog:
(38984, 53)

In [4]:
ml_url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = '/kaggle/working/data/ml-latest-small.zip'

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(ml_url, zip_path)
    rprint("Downloaded.")

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/kaggle/working/data')

ratings = pd.read_csv('/kaggle/working/data/ml-latest-small/ratings.csv')
movies = pd.read_csv('/kaggle/working/data/ml-latest-small/movies.csv')
links = pd.read_csv('/kaggle/working/data/ml-latest-small/links.csv')

rprint("Ratings:", ratings.shape)
rprint("Movies:", movies.shape)
rprint("Links:", links.shape)
rprint()
rprint(ratings.head())
rprint()
rprint(links.head())

Ratings:
(100836, 4)

Movies:
(9742, 3)

Links:
(9742, 3)

userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0

In [6]:
links['tconst'] = 'tt' + links['imdbId'].astype(str).str.zfill(7)
links['content_id'] = 'imdb_' + links['tconst']

rprint(links[['movieId', 'imdbId', 'tconst', 'content_id']].head())

overlap = links[links['content_id'].isin(catalog['content_id'])]

rprint()
rprint(f"MovieLens movies: {len(links)}")
rprint(f"Overlap with our catalog: {len(overlap)} ({len(overlap)/len(links)*100:.1f}%)")

movieId  imdbId     tconst      content_id
0        1  114709  tt0114709  imdb_tt0114709
1        2  113497  tt0113497  imdb_tt0113497
2        3  113228  tt0113228  imdb_tt0113228
3        4  114885  tt0114885  imdb_tt0114885
4        5  113041  tt0113041  imdb_tt0113041

MovieLens movies: 9742

Overlap with our catalog: 8315 (85.4%)

In [8]:
import subprocess

subprocess.run(['pip', 'install', 'scikit-surprise', '--quiet'])

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_split
from surprise import accuracy

reader = Reader(rating_scale = (0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

trainset, testset = surprise_split(data, test_size = 0.2, random_state = 42)

rprint(f"Trainset Size: {trainset.n_ratings}")
rprint(f"Testset Size: {len(testset)}")

Trainset Size: 80668

Testset Size: 20168

In [10]:
model = SVD(
    n_factors = 50,
    n_epochs = 20,
    random_state = 42
)

model.fit(trainset)

predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 0.8775
MAE:  0.6742


In [12]:
def recommend_for_user(user_id, model, ratings, movies, links, catalog, top_n = 10):
    all_movie_ids = ratings['movieId'].unique()
    watched = ratings[ratings['userId'] == user_id]['movieId'].values
    unwatched = [m for m in all_movie_ids if m not in watched]

    predictions = [(m, model.predict(user_id, m).est) for m in unwatched]
    predictions.sort(key=lambda x: x[1], reverse = True)
    top_movie_ids = [m for m, score in predictions[:top_n]]
    top_scores = {m: score for m, score in predictions[:top_n]}

    result = movies[movies['movieId'].isin(top_movie_ids)].copy()
    result['predicted_rating'] = result['movieId'].map(top_scores)
    result = result.merge(links[['movieId', 'content_id']], on = 'movieId', how = 'left')
    result = result.merge(catalog[['content_id', 'rating_normalized', 'popularity_percentile']], on = 'content_id', how = 'left')

    return result.sort_values('predicted_rating', ascending = False)[
        ['title', 'genres', 'predicted_rating', 'rating_normalized', 'popularity_percentile']
    ]

sample_user = 1
rprint("User", sample_user, "rated which (top 5 by rating):")
user_history = ratings[ratings['userId'] == sample_user].merge(movies, on = 'movieId').sort_values('rating', ascending = False)
rprint(user_history[['title', 'genres', 'rating']].head(5))
rprint()

rprint("Model's Recommendation for this User:")
rprint(recommend_for_user(sample_user, model, ratings, movies, links, catalog, top_n=10))

User 1 rated which (top 5 by rating):

title                          genres  \
3              Seven (a.k.a. Se7en) (1995)                Mystery|Thriller   
4               Usual Suspects, The (1995)          Crime|Mystery|Thriller   
6                     Bottle Rocket (1996)  Adventure|Comedy|Crime|Romance   
13  Dumb & Dumber (Dumb and Dumber) (1994)                Adventure|Comedy   
11                    Billy Madison (1995)                          Comedy   

    rating  
3      5.0  
4      5.0  
6      5.0  
13     5.0  
11     5.0

Model's Recommendation for this User:

title  \
0                   Shawshank Redemption, The (1994)   
1         Ghost in the Shell (Kôkaku kidôtai) (1995)   
2     Cinema Paradiso (Nuovo cinema Paradiso) (1989)   
3                          Lawrence of Arabia (1962)   
4                             Rosemary's Baby (1968)   
5         Life Is Beautiful (La Vita è bella) (1997)   
6                        Boondock Saints, The (2000)   
8                         Kiss Kiss Bang Bang (2005)   
7  Amelie (Fabuleux destin d'Amélie Poulain, Le) ...   
9                               Departed, The (2006)   

                          genres  predicted_rating  rating_normalized  \
0                    Crime|Drama          5.000000                9.3   
1               Animation|Sci-Fi          5.000000                NaN   
2                          Drama          5.000000                8.5   
3            Adventure|Drama|War          5.000000                8.3   
4          Drama|Horror|Thriller          5.000000                8.0   
5       Comedy|Drama|Romance|War          5.000000                8.6   
6    Action|Crime|Drama|Thriller          5.000000                7.6   
8  Comedy|Crime|Mystery|Thriller          5.000000                7.4   
7                 Comedy|Romance          4.987584                8.2   
9           Crime|Drama|Thriller          4.981134                8.5   

   popularity_percentile  
0               1.000000  
1                    NaN  
2               0.976359  
3               0.978676  
4               0.968087  
5               0.995336  
6               0.968762  
8               0.966679  
7               0.995600  
9               0.999091

In [14]:
sample_ids = [356, 2571, 260]  
for mid in sample_ids:
    pred_clipped = model.predict(1, mid, clip = True)
    pred_raw = model.predict(1, mid, clip = False)
    rprint(f"movieId {mid}: clipped={pred_clipped.est:.4f} | raw={pred_raw.est:.4f}")

movieId 356: clipped=5.0000 | raw=5.0431

movieId 2571: clipped=4.8731 | raw=4.8731

movieId 260: clipped=4.9690 | raw=4.9690

In [16]:
ghost_check = catalog[catalog['title'].str.contains('Ghost in the Shell', case = False, na = False)]
rprint(ghost_check[['content_id', 'title', 'is_anime', 'year']])

content_id                                              title  \
1086   imdb_tt1219827                                 Ghost in the Shell   
19494  imdb_tt2636124   Ghost in the Shell: Arise - Border 1: Ghost Pain   
34459      anilist_43                                 Ghost in the Shell   
35018     anilist_467            Ghost in the Shell: Stand Alone Complex   
35739     anilist_801    Ghost in the Shell: Stand Alone Complex 2nd GIG   
35864     anilist_468                    Ghost in the Shell 2: Innocence   
35894  anilist_177699                             THE GHOST IN THE SHELL   
36931    anilist_4672                             Ghost in the Shell 2.0   
36948    anilist_1566  Ghost in the Shell: Stand Alone Complex - Soli...   
37413   anilist_17187    Ghost in the Shell: Arise - Border:1 Ghost Pain   
37787  anilist_106154                       Ghost in the Shell: SAC_2045   
37850   anilist_19191  Ghost in the Shell: Arise - Border:2 Ghost Whi...   
38025   anilist_21057                  Ghost in the Shell: The New Movie   
38028   anilist_19193   Ghost in the Shell: Arise - Border:3 Ghost Tears   
38173   anilist_19195  Ghost in the Shell: Arise - Border:4 Ghost Sta...   
38237   anilist_21056  Ghost in the Shell Arise: Alternative Architec...   
38686    anilist_2449  Ghost in the Shell: Stand Alone Complex - The ...   

       is_anime    year  
1086      False  2017.0  
19494     False  2013.0  
34459      True  1995.0  
35018      True  2002.0  
35739      True  2004.0  
35864      True  2004.0  
35894      True  2026.0  
36931      True  2008.0  
36948      True  2006.0  
37413      True  2013.0  
37787      True  2020.0  
37850      True  2013.0  
38025      True  2015.0  
38028      True  2014.0  
38173      True  2014.0  
38237      True  2015.0  
38686      True  2005.0

In [17]:
def recommend_for_user(user_id, model, ratings, movies, links, catalog, top_n = 10):
    all_movie_ids = ratings['movieId'].unique()
    watched = ratings[ratings['userId'] == user_id]['movieId'].values
    unwatched = [m for m in all_movie_ids if m not in watched]

    predictions = [(m, model.predict(user_id, m, clip=False).est) for m in unwatched]
    predictions.sort(key=lambda x: x[1], reverse = True)
    top_movie_ids = [m for m, score in predictions[:top_n]]
    top_scores = {m: score for m, score in predictions[:top_n]}

    result = movies[movies['movieId'].isin(top_movie_ids)].copy()
    result['predicted_rating'] = result['movieId'].map(top_scores)
    result = result.merge(links[['movieId', 'content_id']], on = 'movieId', how = 'left')
    result = result.merge(catalog[['content_id', 'rating_normalized', 'popularity_percentile']], on = 'content_id', how = 'left')

    return result.sort_values('predicted_rating', ascending = False)[
        ['title', 'genres', 'predicted_rating', 'rating_normalized', 'popularity_percentile']
    ]

In [18]:
import re

def clean_ml_title(t):
    match = re.match(r'^(.*?)(?:\s\([^)]*\))?\s\((\d{4})\)$', t.strip())
    if match:
        title = re.sub(r'[^a-z0-9]', '', match.group(1).lower())
        return title, int(match.group(2))
    return re.sub(r'[^a-z0-9]', '', t.lower()), None

movies['clean_title'], movies['ml_year'] = zip(*movies['title'].apply(clean_ml_title))

anime_lookup = catalog[catalog['is_anime']].copy()
anime_lookup['clean_title'] = anime_lookup['title'].str.lower().str.replace(r'[^a-z0-9]', '', regex = True)

anime_fallback = movies.merge(anime_lookup[['content_id', 'clean_title', 'year']], on = 'clean_title', how = 'inner')
anime_fallback = anime_fallback[anime_fallback['ml_year'] == anime_fallback['year']]
anime_fallback = anime_fallback[['movieId', 'content_id']].drop_duplicates('movieId')

rprint("Fallback anime matches found:", len(anime_fallback))
rprint(anime_fallback.head())

Fallback anime matches found: 83

movieId    content_id
0       741    anilist_43
1      1274    anilist_47
9      3000   anilist_164
10     3054   anilist_528
14     4446  anilist_1361

In [19]:
links_updated = links.merge(anime_fallback, on = 'movieId', how = 'left', suffixes = ('', '_fallback'))
links_updated['content_id'] = links_updated['content_id'].fillna(links_updated['content_id_fallback'])
links_updated = links_updated.drop(columns=['content_id_fallback'])

rprint("Total content_id resolved:", links_updated['content_id'].notna().sum(), "/", len(links_updated))

Total content_id resolved: 9742 / 9742

In [20]:
in_catalog_before = links['content_id'].isin(catalog['content_id'])
rprint("Before fallback - resolved against catalog:", in_catalog_before.sum(), "/", len(links))

links_updated = links.merge(anime_fallback, on='movieId', how='left', suffixes=('', '_fallback'))

needs_fallback = ~links_updated['content_id'].isin(catalog['content_id'])
links_updated.loc[needs_fallback, 'content_id'] = links_updated.loc[needs_fallback, 'content_id_fallback'].combine_first(
    links_updated.loc[needs_fallback, 'content_id']
)
links_updated = links_updated.drop(columns = ['content_id_fallback'])

in_catalog_after = links_updated['content_id'].isin(catalog['content_id'])
rprint("After fallback - resolved against catalog:", in_catalog_after.sum(), "/", len(links_updated))

Before fallback - resolved against catalog: 8315 / 9742

After fallback - resolved against catalog: 8387 / 9742

In [21]:
rprint("User 1 recommendations (fixed):")
rprint(recommend_for_user(1, model, ratings, movies, links_updated, catalog, top_n = 10))

User 1 recommendations (fixed):

title  \
2     Cinema Paradiso (Nuovo cinema Paradiso) (1989)   
5         Life Is Beautiful (La Vita è bella) (1997)   
0                   Shawshank Redemption, The (1994)   
3                          Lawrence of Arabia (1962)   
8                         Kiss Kiss Bang Bang (2005)   
4                             Rosemary's Baby (1968)   
1         Ghost in the Shell (Kôkaku kidôtai) (1995)   
6                        Boondock Saints, The (2000)   
7  Amelie (Fabuleux destin d'Amélie Poulain, Le) ...   
9                               Departed, The (2006)   

                          genres  predicted_rating  rating_normalized  \
2                          Drama          5.176636                8.5   
5       Comedy|Drama|Romance|War          5.117477                8.6   
0                    Crime|Drama          5.103252                9.3   
3            Adventure|Drama|War          5.080314                8.3   
8  Comedy|Crime|Mystery|Thriller          5.057419                7.4   
4          Drama|Horror|Thriller          5.045840                8.0   
1               Animation|Sci-Fi          5.033436                8.0   
6    Action|Crime|Drama|Thriller          5.012669                7.6   
7                 Comedy|Romance          4.987584                8.2   
9           Crime|Drama|Thriller          4.981134                8.5   

   popularity_percentile  
2               0.976359  
5               0.995336  
0               1.000000  
3               0.978676  
8               0.966679  
4               0.968087  
1               0.925169  
6               0.968762  
7               0.995600  
9               0.999091

In [22]:
import pickle

with open('/kaggle/working/processed/svd_model_v1.pkl', 'wb') as f:
    pickle.dump(model, f)

links_updated.to_csv('/kaggle/working/processed/movielens_links_resolved.csv', index = False)

ratings.to_csv('/kaggle/working/processed/movielens_ratings.csv', index = False)
movies.to_csv('/kaggle/working/processed/movielens_movies.csv', index = False)

rprint("Saved all Stage 3 artifacts.")

Saved all Stage 3 artifacts.

In [23]:
from huggingface_hub import HfApi, login

login(token = HF_TOKEN)

api = HfApi()

In [24]:
api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/svd_model_v1.pkl',
    path_in_repo = 'svd_model_v1.pkl',
    repo_id = 'Subhadip007/UERP_Model',
    repo_type = 'model',
)

for fname in ['movielens_links_resolved.csv', 'movielens_ratings.csv', 'movielens_movies.csv']:
    api.upload_file(
        path_or_fileobj = f'/kaggle/working/processed/{fname}',
        path_in_repo = fname,
        repo_id = 'Subhadip007/UERP_Dataset',
        repo_type = 'dataset',
    )

rprint("Pushed to HF Hub — model to UERP_Model, data to UERP_Dataset")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to HF Hub — model to UERP_Model, data to UERP_Dataset

In [25]:
ml25_url = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
zip_path_25m = '/kaggle/working/data/ml-25m.zip'

if not os.path.exists(zip_path_25m):
    rprint("Downloading ml-25m (this is a large file, may take a few minutes)...")
    urllib.request.urlretrieve(ml25_url, zip_path_25m)
    rprint("Downloaded.")

with zipfile.ZipFile(zip_path_25m, 'r') as z:
    z.extractall('/kaggle/working/data')

rprint("Extracted.")

Downloading ml-25m (this is a large file, may take a few minutes)...

Downloaded.

Extracted.

In [26]:
ratings_25m = pd.read_csv(
    '/kaggle/working/data/ml-25m/ratings.csv',
    dtype = {'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'},
)
movies_25m = pd.read_csv('/kaggle/working/data/ml-25m/movies.csv')
links_25m = pd.read_csv('/kaggle/working/data/ml-25m/links.csv')

rprint("Ratings:", ratings_25m.shape)
rprint("Movies:", movies_25m.shape)
rprint("Links:", links_25m.shape)
rprint()
rprint("Unique users:", ratings_25m['userId'].nunique())
rprint("Unique movies rated:", ratings_25m['movieId'].nunique())

Ratings:
(25000095, 4)

Movies:
(62423, 3)

Links:
(62423, 3)

Unique users: 162541

Unique movies rated: 59047

In [27]:
links_25m['tconst'] = 'tt' + links_25m['imdbId'].astype(str).str.zfill(7)
links_25m['content_id'] = 'imdb_' + links_25m['tconst']

in_catalog = links_25m['content_id'].isin(catalog['content_id'])
rprint("Direct match against catalog:", in_catalog.sum(), "/", len(links_25m))

movies_25m['clean_title'], movies_25m['ml_year'] = zip(*movies_25m['title'].apply(clean_ml_title))
anime_fallback_25m = movies_25m.merge(anime_lookup[['content_id', 'clean_title', 'year']], on='clean_title', how = 'inner')
anime_fallback_25m = anime_fallback_25m[anime_fallback_25m['ml_year'] == anime_fallback_25m['year']]
anime_fallback_25m = anime_fallback_25m[['movieId', 'content_id']].drop_duplicates('movieId')

links_25m_updated = links_25m.merge(anime_fallback_25m, on = 'movieId', how = 'left', suffixes = ('', '_fallback'))
needs_fallback = ~links_25m_updated['content_id'].isin(catalog['content_id'])
links_25m_updated.loc[needs_fallback, 'content_id'] = links_25m_updated.loc[needs_fallback, 'content_id_fallback'].combine_first(
    links_25m_updated.loc[needs_fallback, 'content_id']
)
links_25m_updated = links_25m_updated.drop(columns = ['content_id_fallback'])

resolved_final = links_25m_updated['content_id'].isin(catalog['content_id'])
rprint("After fallback, resolved:", resolved_final.sum(), "/", len(links_25m_updated))

Direct match against catalog: 18114 / 62423

After fallback, resolved: 18325 / 62423

In [28]:
import time

sample_ratings = ratings_25m.sample(n = 2_000_000, random_state = 42)
reader = Reader(rating_scale=(0.5, 5.0))
data_sample = Dataset.load_from_df(sample_ratings[['userId', 'movieId', 'rating']], reader)
trainset_sample = data_sample.build_full_trainset()

start = time.time()
model_test = SVD(n_factors = 50, n_epochs = 20, random_state = 42)
model_test.fit(trainset_sample)
elapsed = time.time() - start

rprint(f"2M ratings training time: {elapsed:.1f} seconds")
rprint(f"Estimated time for full 25M: ~{elapsed * 12.5 / 60:.1f} minutes")

2M ratings training time: 42.1 seconds

Estimated time for full 25M: ~8.8 minutes

In [29]:
resolved_movie_ids = set(links_25m_updated[resolved_final]['movieId'])

ratings_in_catalog = ratings_25m['movieId'].isin(resolved_movie_ids).sum()
total_ratings = len(ratings_25m)

rprint(f"Ratings on catalog-matched movies: {ratings_in_catalog:,} / {total_ratings:,} ({ratings_in_catalog/total_ratings*100:.1f}%)")

Ratings on catalog-matched movies: 24,454,508 / 25,000,095 (97.8%)

In [30]:
reader = Reader(rating_scale = (0.5, 5.0))
data_full = Dataset.load_from_df(ratings_25m[['userId', 'movieId', 'rating']], reader)
trainset_full = data_full.build_full_trainset()

print("Training on full trainset:", trainset_full.n_ratings, "ratings,", trainset_full.n_users, "users,", trainset_full.n_items, "items")

start = time.time()
model_25m = SVD(n_factors = 50, n_epochs = 20, random_state = 42)
model_25m.fit(trainset_full)
elapsed = time.time() - start
rprint(f"Training time: {elapsed/60:.1f} minutes")

Training on full trainset: 25000095 ratings, 162541 users, 59047 items


Training time: 2.8 minutes

In [31]:
from surprise.model_selection import train_test_split as surprise_split
from surprise import accuracy

trainset_eval, testset_eval = surprise_split(data_full, test_size = 0.1, random_state = 42)

model_eval = SVD(n_factors = 50, n_epochs = 20, random_state = 42)
model_eval.fit(trainset_eval)
predictions_eval = model_eval.test(testset_eval)

rmse_25m = accuracy.rmse(predictions_eval)
mae_25m = accuracy.mae(predictions_eval)

RMSE: 0.7728
MAE:  0.5832


In [32]:
import pickle

with open('/kaggle/working/processed/svd_model_25m.pkl', 'wb') as f:
    pickle.dump(model_25m, f)

links_25m_updated.to_csv('/kaggle/working/processed/movielens_25m_links_resolved.csv', index = False)

movies_25m.to_csv('/kaggle/working/processed/movielens_25m_movies.csv', index = False)

rprint("Saved 25M artifacts locally.")

Saved 25M artifacts locally.

In [33]:
api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/svd_model_25m.pkl',
    path_in_repo = 'svd_model_25m.pkl',
    repo_id = 'Subhadip007/UERP_Model',
    repo_type = 'model',
)

for fname in ['movielens_25m_links_resolved.csv', 'movielens_25m_movies.csv']:
    api.upload_file(
        path_or_fileobj = f'/kaggle/working/processed/{fname}',
        path_in_repo = fname,
        repo_id = 'Subhadip007/UERP_Dataset',
        repo_type = 'dataset',
    )

rprint("25M model + data pushed to HF Hub.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

25M model + data pushed to HF Hub.

In [34]:
def recommend_for_user_25m(user_id, model, ratings, movies, links, catalog, top_n = 10):
    all_movie_ids = ratings['movieId'].unique()
    watched = ratings[ratings['userId'] == user_id]['movieId'].values
    unwatched_sample = np.setdiff1d(all_movie_ids, watched)

    predictions = [(m, model.predict(user_id, m, clip = False).est) for m in unwatched_sample]
    predictions.sort(key = lambda x: x[1], reverse = True)
    top_movie_ids = [m for m, score in predictions[:top_n]]
    top_scores = {m: score for m, score in predictions[:top_n]}

    result = movies[movies['movieId'].isin(top_movie_ids)].copy()
    result['predicted_rating'] = result['movieId'].map(top_scores)
    result = result.merge(links[['movieId', 'content_id']], on = 'movieId', how = 'left')
    result = result.merge(catalog[['content_id', 'rating_normalized', 'popularity_percentile']], on = 'content_id', how = 'left')

    return result.sort_values('predicted_rating', ascending = False)[
        ['title', 'genres', 'predicted_rating', 'rating_normalized', 'popularity_percentile']
    ]

sample_user_25m = ratings_25m['userId'].iloc[0]
rprint("Sample user:", sample_user_25m)
rprint()
rprint("Unke history se (top rated):")
history = ratings_25m[ratings_25m['userId'] == sample_user_25m].merge(movies_25m, on = 'movieId').sort_values('rating', ascending = False)
rprint(history[['title', 'genres', 'rating']].head(5))
rprint()
rprint("Recommendations:")
rprint(recommend_for_user_25m(sample_user_25m, model_25m, ratings_25m, movies_25m, links_25m_updated, catalog, top_n = 10))

Sample user: 1

Unke history se (top rated):

title  \
0                                 Pulp Fiction (1994)   
2    Three Colors: Blue (Trois couleurs: Bleu) (1993)   
3                                  Underground (1995)   
18  Saragossa Manuscript, The (Rekopis znaleziony ...   
19                   Run Lola Run (Lola rennt) (1998)   

                         genres  rating  
0   Comedy|Crime|Drama|Thriller     5.0  
2                         Drama     5.0  
3              Comedy|Drama|War     5.0  
18      Adventure|Drama|Mystery     5.0  
19                 Action|Crime     5.0

Recommendations:

title                       genres  \
0                   Godfather, The (1972)                  Crime|Drama   
4                  American Beauty (1999)                Drama|Romance   
8           The Heart of the World (2000)                Drama|Fantasy   
2              Clockwork Orange, A (1971)  Crime|Drama|Sci-Fi|Thriller   
7                Kill Bill: Vol. 2 (2004)        Action|Drama|Thriller   
3          Godfather: Part II, The (1974)                  Crime|Drama   
6                Kill Bill: Vol. 1 (2003)        Action|Crime|Thriller   
9                      I, Claudius (1976)                        Drama   
5                       Fight Club (1999)  Action|Crime|Drama|Thriller   
1  One Flew Over the Cuckoo's Nest (1975)                        Drama   

   predicted_rating  rating_normalized  popularity_percentile  
0          4.584098                9.2               0.999707  
4          4.518795                8.3               0.998592  
8          4.488953                7.6               0.230531  
2          4.486850                8.2               0.996656  
7          4.467675                8.0               0.995864  
3          4.462422                9.0               0.999003  
6          4.449437                8.2               0.998651  
9          4.406230                8.8               0.730341  
5          4.403558                8.8               0.999853  
1          4.394257                8.6               0.997859

In [35]:
heart_movie_id = movies_25m[movies_25m['title'].str.contains('Heart of the World', case = False, na = False)]['movieId'].values[0]
rating_count = (ratings_25m['movieId'] == heart_movie_id).sum()

rprint(f"'The Heart of the World' — total ratings in dataset: {rating_count}")

'The Heart of the World' — total ratings in dataset: 33

In [36]:
movie_rating_counts = ratings_25m.groupby('movieId').size()

def recommend_for_user_25m(user_id, model, ratings, movies, links, catalog, 
                            min_rating_count = 30, top_n = 10):
    all_movie_ids = ratings['movieId'].unique()
    watched = ratings[ratings['userId'] == user_id]['movieId'].values
    unwatched = np.setdiff1d(all_movie_ids, watched)

    reliable_items = movie_rating_counts[movie_rating_counts >= min_rating_count].index
    unwatched = np.intersect1d(unwatched, reliable_items)

    predictions = [(m, model.predict(user_id, m, clip = False).est) for m in unwatched]
    predictions.sort(key = lambda x: x[1], reverse = True)
    top_movie_ids = [m for m, score in predictions[:top_n]]
    top_scores = {m: score for m, score in predictions[:top_n]}

    result = movies[movies['movieId'].isin(top_movie_ids)].copy()
    result['predicted_rating'] = result['movieId'].map(top_scores)
    result = result.merge(links[['movieId', 'content_id']], on = 'movieId', how = 'left')
    result = result.merge(catalog[['content_id', 'rating_normalized', 'popularity_percentile']], on = 'content_id', how = 'left')

    return result.sort_values('predicted_rating', ascending = False)[
        ['title', 'genres', 'predicted_rating', 'rating_normalized', 'popularity_percentile']
    ]

rprint(recommend_for_user_25m(1, model_25m, ratings_25m, movies_25m, links_25m_updated, catalog, min_rating_count = 30, top_n = 10))

title                       genres  \
0                   Godfather, The (1972)                  Crime|Drama   
4                  American Beauty (1999)                Drama|Romance   
8           The Heart of the World (2000)                Drama|Fantasy   
2              Clockwork Orange, A (1971)  Crime|Drama|Sci-Fi|Thriller   
7                Kill Bill: Vol. 2 (2004)        Action|Drama|Thriller   
3          Godfather: Part II, The (1974)                  Crime|Drama   
6                Kill Bill: Vol. 1 (2003)        Action|Crime|Thriller   
9                      I, Claudius (1976)                        Drama   
5                       Fight Club (1999)  Action|Crime|Drama|Thriller   
1  One Flew Over the Cuckoo's Nest (1975)                        Drama   

   predicted_rating  rating_normalized  popularity_percentile  
0          4.584098                9.2               0.999707  
4          4.518795                8.3               0.998592  
8          4.488953                7.6               0.230531  
2          4.486850                8.2               0.996656  
7          4.467675                8.0               0.995864  
3          4.462422                9.0               0.999003  
6          4.449437                8.2               0.998651  
9          4.406230                8.8               0.730341  
5          4.403558                8.8               0.999853  
1          4.394257                8.6               0.997859

In [37]:
rprint(recommend_for_user_25m(1, model_25m, ratings_25m, movies_25m, links_25m_updated, catalog, min_rating_count = 100, top_n = 10))

title                       genres  \
0                   Godfather, The (1972)                  Crime|Drama   
4                  American Beauty (1999)                Drama|Romance   
2              Clockwork Orange, A (1971)  Crime|Drama|Sci-Fi|Thriller   
7                Kill Bill: Vol. 2 (2004)        Action|Drama|Thriller   
3          Godfather: Part II, The (1974)                  Crime|Drama   
6                Kill Bill: Vol. 1 (2003)        Action|Crime|Thriller   
5                       Fight Club (1999)  Action|Crime|Drama|Thriller   
1  One Flew Over the Cuckoo's Nest (1975)                        Drama   
9                            Black Mirror           (no genres listed)   
8                 Twelve Angry Men (1954)                        Drama   

   predicted_rating  rating_normalized  popularity_percentile  
0          4.584098                9.2               0.999707  
4          4.518795                8.3               0.998592  
2          4.486850                8.2               0.996656  
7          4.467675                8.0               0.995864  
3          4.462422                9.0               0.999003  
6          4.449437                8.2               0.998651  
5          4.403558                8.8               0.999853  
1          4.394257                8.6               0.997859  
9          4.385312                NaN                    NaN  
8          4.371336                NaN                    NaN

In [38]:
debug_titles = movies_25m[movies_25m['title'].str.contains('Twelve Angry Men|^Black Mirror', case = False, na = False, regex = True)]
rprint(debug_titles)
rprint()

debug_ids = debug_titles['movieId'].tolist()
debug_links = links_25m_updated[links_25m_updated['movieId'].isin(debug_ids)]
rprint(debug_links)
rprint()

for _, row in debug_links.iterrows():
    exists = row['content_id'] in catalog['content_id'].values
    rprint(f"movieId = {row['movieId']} | content_id = {row['content_id']} | in catalog: {exists}")

movieId                              title  \
24372   121320            Twelve Angry Men (1963)   
48110   175981            Twelve Angry Men (1954)   
48413   176601                       Black Mirror   
57716   196997  Black Mirror: Bandersnatch (2018)   

                              genres              clean_title  ml_year  
24372                          Drama           twelveangrymen   1963.0  
48110                          Drama           twelveangrymen   1954.0  
48413             (no genres listed)              blackmirror      NaN  
57716  Drama|Mystery|Sci-Fi|Thriller  blackmirrorbandersnatch   2018.0

movieId   imdbId    tmdbId     tconst      content_id
24372   121320    57723  269165.0  tt0057723  imdb_tt0057723
48110   175981   122737  269981.0  tt0122737  imdb_tt0122737
48413   176601  2492564  452830.0  tt2492564  imdb_tt2492564
57716   196997  9495224  569547.0  tt9495224  imdb_tt9495224

movieId = 121320 | content_id = imdb_tt0057723 | in catalog: False

movieId = 175981 | content_id = imdb_tt0122737 | in catalog: False

movieId = 176601 | content_id = imdb_tt2492564 | in catalog: False

movieId = 196997 | content_id = imdb_tt9495224 | in catalog: True

In [39]:
%%writefile /kaggle/working/processed/recommender_cf.py

import numpy as np

def recommend_for_user(user_id, model, ratings, movies, links, catalog, movie_rating_counts,
                        min_rating_count = 100, top_n = 10):
    all_movie_ids = ratings['movieId'].unique()
    watched = ratings[ratings['userId'] == user_id]['movieId'].values
    unwatched = np.setdiff1d(all_movie_ids, watched)

    reliable_items = movie_rating_counts[movie_rating_counts >= min_rating_count].index
    unwatched = np.intersect1d(unwatched, reliable_items)

    predictions = [(m, model.predict(user_id, m, clip=False).est) for m in unwatched]
    predictions.sort(key=lambda x: x[1], reverse = True)
    top_movie_ids = [m for m, score in predictions[:top_n]]
    top_scores = {m: score for m, score in predictions[:top_n]}

    result = movies[movies['movieId'].isin(top_movie_ids)].copy()
    result['predicted_rating'] = result['movieId'].map(top_scores)
    result = result.merge(links[['movieId', 'content_id']], on = 'movieId', how = 'left')
    result = result.merge(catalog[['content_id', 'rating_normalized', 'popularity_percentile']], on = 'content_id', how = 'left')

    return result.sort_values('predicted_rating', ascending = False)

Writing /kaggle/working/processed/recommender_cf.py
